# Golang RAG 辅导助手（第 5 周解答）

## 练习目标

用 **RAG（Retrieval Augmented Generation，检索增强生成）** 做一个 Golang 知识辅导聊天机器人：

- 从本地 `knowledge_docs` 加载 Markdown 文档
- 切块（chunk）→ 向量嵌入（embedding）→ 存入 **Chroma** 向量库
- 用户提问时先检索相关片段，再交给 LLM 基于上下文作答
- 用 **Gradio** `ChatInterface` 提供对话界面

## 和本课第 5 周的关系

| 本课概念 | 本笔记本里你会看到 |
|----------|-------------------|
| Document Loader | `DirectoryLoader` + `TextLoader` |
| Text Splitter | `RecursiveCharacterTextSplitter` |
| Embeddings | `OpenAIEmbeddings`（经 OpenRouter） |
| Vector Store | `Chroma.from_documents` |
| Retriever + LLM | `as_retriever()` + `ChatOpenAI` |
| UI | `gr.ChatInterface` |

## 怎么跑

1. 准备 `.env`：设置 `OPENROUTER_API_KEY`
2. 确保同目录有 `knowledge_docs/*.md`
3. 从上到下依次运行单元格；最后一格会启动 Gradio


In [16]:
# ========== 导入：RAG 管线 + Gradio UI 所需工具 ==========

# 标准库 os：读环境变量、判断路径是否存在
import os
# 标准库 glob：按通配符枚举知识库里的 Markdown 文件
import glob
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量（Environment Variables）
from dotenv import load_dotenv
# LangChain OpenAI 嵌入：把文本变成向量（经 OpenRouter 的兼容接口）
from langchain_openai import OpenAIEmbeddings
# LangChain Chat 模型：调用对话 LLM
from langchain_openai import ChatOpenAI
# Chroma：本地持久化向量库
from langchain_chroma import Chroma
# 文档加载器：按目录批量读 Markdown
from langchain_community.document_loaders import DirectoryLoader, TextLoader
# 递归字符切块器：按长度/分隔符切文档，并保留重叠（overlap）
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 消息类型：SystemMessage（系统角色）与 HumanMessage（用户角色）
from langchain_core.messages import SystemMessage, HumanMessage
# Gradio：快速搭聊天 Web UI
import gradio as gr


In [10]:
# ========== 配置：模型名、向量库目录、API Key ==========

# 对话模型 id（经 OpenRouter 路由；字符串保持原样）
MODEL = "gpt-4.1-nano"
# Chroma 持久化目录名：向量会写到磁盘，下次可复用
db_name = "golang_knowledge_db"
# 加载 .env；override=True 表示覆盖已有同名环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenRouter API Key
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
# 有密钥则打印前缀做冒烟检查（不打印完整密钥）
if openrouter_api_key:
    print(f"Open Router API Key exists and begins {openrouter_api_key[:3]}")
else:
    # 未设置时给出提示（错误文案字符串保持原样）
    print("OpenRouter API Key not set")


Open Router API Key exists and begins sk-


In [11]:
# ========== 可选探查：用 glob 数一下知识库文件，并拼成大字符串 ==========
# 注意：后面真正进 RAG 的是 DirectoryLoader；本格偏「看看有多少文档」

# 知识库通配路径：knowledge_docs 下所有 .md
knowledge_base_path = "knowledge_docs/*.md"
# recursive=True：允许匹配子目录（本通配本身已指向一层）
files = glob.glob(knowledge_base_path, recursive=True)
# 打印找到的文件数量
print(f"Found {len(files)} files in the knowledge docs")

# 累积全文的缓冲区（演示用）
entire_knowledge_docs = ""

# 逐文件读取并拼接
for file_path in files:
    # 以 UTF-8 打开，避免中文/特殊字符乱码
    with open(file_path, 'r', encoding='utf-8') as f:
        # 追加文件正文
        entire_knowledge_docs += f.read()
        # 文件之间加空行分隔
        entire_knowledge_docs += "\n\n"



Found 3 files in the knowledge docs


In [12]:
# ========== 正式加载：DirectoryLoader → LangChain Document 列表 ==========

# 扫描 knowledge_docs 下全部 .md；TextLoader + utf-8 解码
loader = DirectoryLoader("knowledge_docs", glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
# load()：读盘并变成 Document（含 page_content 与 metadata）
documents = loader.load()
# 为每个文档补 metadata：用文件名（去扩展名）当作 doc_type，便于日后过滤
for doc in documents:
    doc.metadata["doc_type"] = os.path.splitext(os.path.basename(doc.metadata.get("source", "")))[0]


In [13]:
# ========== 切块 + 嵌入 + 写入 Chroma 向量库 ==========

# 切块器：每块约 1000 字符，块间重叠 200，减少边界处语义断裂
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# 把 Document 列表切成更小的 chunks
chunks = text_splitter.split_documents(documents)


# OpenRouter 兼容的嵌入模型客户端（model / base_url / api_key 保持原样）
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-large",
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
)

# 若本地已有同名向量库，先删掉旧 collection，避免新旧数据混在一起
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

# 从 chunks 建库并持久化到 db_name 目录
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
# 打印 collection 中向量条数，确认写入成功
print(f"Vectorstore created with {vectorstore._collection.count()} documents")


Vectorstore created with 24 documents


In [14]:
# ========== System Prompt 模板：规定「如何用检索到的 Context 答题」==========
# 注意：花括号 {context} 留给后面 .format() 填入检索片段；英文 prompt 正文不翻译

SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly tutor helping learners master fundamental Go (Golang) concepts.
You explain clearly with examples, encourage good practices, and adapt to the learner's level.
When answering, use the given context from the knowledge base. Cite or reference it when relevant.
If the context doesn't contain the answer, say so and offer to clarify or point to other resources.
Context:
{context}
"""


In [15]:
# ========== 检索器 + 对话模型：RAG 的「查」与「答」两端 ==========

# 默认相似度检索器（从 vectorstore 取相关文档）
retriever = vectorstore.as_retriever()
# ChatOpenAI：temperature=0 更稳；走 OpenRouter 的 base_url 与密钥
llm = ChatOpenAI(temperature=0, model_name=MODEL, base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)


In [17]:
# ========== RAG 核心函数：检索 → 填 prompt → 调 LLM → 返回文本 ==========

def answer_question(question: str, history):
    # history 由 Gradio ChatInterface 传入；本实现未用对话历史，只答当前问题
    # 用问题做查询，取回相关 Document 列表
    docs = retriever.invoke(question)
    # 把各块正文用空行拼成一段 context
    context = "\n\n".join(doc.page_content for doc in docs)
    # 把 context 填进 system prompt 模板
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    # 系统消息 + 用户问题，一次性 invoke
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    # 只把模型回复正文交给 Gradio 显示
    return response.content


In [ ]:
# ========== 启动 Gradio 聊天界面：回调即上面的 answer_question ==========
gr.ChatInterface(answer_question).launch()
